# Generators LAB

Podstawą wszystkich rozwiązań typu RAG (ang. Retrieval Augmented Generation) jest komponent wyszukiwania. Wyszukiwanie polega na identyfikacji odpowiednich dokumentów, w naszym RAGOwym przypadku takich dokumentów, które potencjalnie zawierają informacje kluczowe do odpowiedzenia na postawione pytanie. Definicja dokumentu jest też rzeczą płynną, raz to może być paragraf, a raz strona, czy rozdział, zależy to od zadania, oraz modeli jakie używamy.

Po realizacji procesu wyszukania top N dokumentów (np. N=100), bardzo często dokonuje się re-ranking aby wybrać top5-10 najlepszych i takie wyselekcjonowane dokumenty tworzą tzw. kontekst. Kontekst to konkatenecja tych top5-10 najlepszych dokumentów. Mówiąc konkatentacja mam na myśli wylistowanie tych dokumentów w formie jednego tekstu.

Na wykładzie starałem się wyraźnie zwrócić uwagę na to, że taki kontekst staje się kluczowym elementem prompta wysyłanego do modelu LLM pełniącego rolę generatora. Generator to LLM pełniący rolę ostatniegu komponentu w RAG architekturze oraz produkującego finalną odpowiedź w języku naturalnym. Generator to model LLM, który po otrzymaniu na wejściu, w prompcie, pytanie i kontekst generuje odpowiedź w języku naturalnym na zadane pytanie w dostarczonym ujętym kontekście informacyjnym. Reasumując, prompt do generatora zawiera pytanie i kontekst, a completion to odpowiedź generatora.

Przykładowey szablon prompta do generatora, który posiada listę dokumentów budujących kontekst informacyjny, oraz instrukcje, a na końcu włąsciwe pytanie.


```
Numerowana lista dokumentów jest poniżej:
---------------------
<results>{% for doc in docs %}
Dokument: {{ loop.index0 }}
{{ doc }}
{% endfor %}</results>
---------------------
Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

Pytanie: {{ question }}

```


Do pracy wymagamy zainstalowania pakietów - `transformers` oraz `datasets`


In [1]:
!pip install -U transformers datasets

Zanim przejdziemy do zdefiniowania szczegółowych zadań sprawdźmy jak użyć powyższy szablon prompta RAGowego do istniejącego LLMa, który ma kompetencje bycia generatorem w RAGach.


Przykładowe użycie generatora na statycznych danych z wykorzystaniem pipeline z transformers oraz modeli PLLuM LLama 8b

In [2]:
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

In [3]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"

# Prompt do generatora z kontekstem opartym na przykładowych wyszukanych tekstach
prepared_prompt = """
Numerowana lista dokumentów jest poniżej:
---------------------

Dokument: 0

Uzyskaj paszport - usługa dla osoby dorosłej
Co musisz przygotować
twoje jedno aktualne, kolorowe zdjęcie - zrób je u fotografa nie wcześniej niż 6 miesięcy przed złożeniem wniosku. Jeśli chcesz je zrobić samodzielnie - sprawdź, jak wykonać zdjęcie do paszportu,
dowód opłaty za paszport (po złożeniu wniosku opłata nie podlega zwrotowi),
jeśli przysługuje ci prawo do zniżki lub zwolnienia z opłaty - dokument, który to potwierdzi, na przykład ważna legitymacja studencka (szczegóły znajdziesz w sekcji Zwolnienia i zniżki),
ważny paszport, którego obecnie używasz. Jeśli nie masz ważnego paszportu - weź ze sobą ważny dowód osobisty,
jeśli twoje nazwisko zostało zmienione po ślubie za granicą - skrócony lub zupełny odpis polskiego aktu małżeństwa.


Dokument: 1

Uzyskaj paszport - usługa dla osoby dorosłej
Jak odbierzesz dokument

Osobiście w punkcie paszportowym - tam, gdzie złożysz wniosek.

Weź ze sobą:

ważny dokument tożsamości,
aktualny dokument paszportowy - jeśli masz,
poprzedni paszport - jeśli masz. Zostanie on anulowany.
Przy odbiorze paszportu sprawdź - czy twoje dane osobowe i biometryczne są poprawne.


---------------------
Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: "Nie udało mi się odnaleźć odpowiedzi na pytanie".

Pytanie: Jak odebrać paszport?"""

messages = [
    {"role": "system", "content": "You are an administration chatbot assistant!"},
    {"role": "user", "content": prepared_prompt},
]

In [4]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"dtype": torch.float16}, #float16 if not ampere/hopper gpu
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [5]:
pipeline.model.eval()
with torch.inference_mode():
    start = time.perf_counter()
    outputs = pipeline(
        messages,
        max_new_tokens=256,
    )
    end = time.perf_counter()
print(outputs[0]["generated_text"][-1])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się osobiście do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy posiadać ważny dokument tożsamości, a także aktualny dokument paszportowy, jeśli się go posiada [1]. Ponadto, należy mieć przy sobie poprzedni paszport, który zostanie anulowany [1]. Przy odbiorze paszportu należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


Czas

In [6]:
print(f"Total time: {end-start:f}s")

Total time: 129.825174s


Zużycie VRAM

In [7]:
!nvidia-smi

Wed May 13 03:03:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             63W /   70W |   14489MiB /  15360MiB |     42%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Zużycie RAM

In [8]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       6.8Gi       1.4Gi        59Mi       4.4Gi       5.5Gi
Swap:             0B          0B          0B


Powyżej pokazałem jak użyć modelu PLLuM LLama 8b Instruct (CYFRAGOVPL/Llama-PLLuM-8B-instruct) do generacji na bazie prompta zawierającego kontekst (lista dokumentów) i pytanie. Teraz czas na twoje zadania.

ZADANIE #1 - Sprawdź jak działa kod wyżej umieszczony na modelu typu chat - CYFRAGOVPL/Llama-PLLuM-8B-chat, czy widzisz różnice, oraz opisz co zaobserwowałeś jak device_map zmienisz z "auto" na "cuda", wytłumacz dlaczego tak się dzieje?

In [9]:
import gc
del pipeline, outputs
gc.collect()
torch.cuda.empty_cache()

In [10]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-chat"

In [11]:
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    model_kwargs={"dtype": torch.float16},
    device_map="auto",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [12]:
pipeline.model.eval()
with torch.inference_mode():
    start = time.perf_counter()
    outputs = pipeline(
        messages,
        max_new_tokens=256,
    )
    end = time.perf_counter()
print(outputs[0]["generated_text"][-1])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'role': 'assistant', 'content': 'Aby odebrać paszport, należy udać się osobiście do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy mieć przy sobie ważny dokument tożsamości, aktualny dokument paszportowy (jeśli jest) oraz poprzedni paszport (jeśli jest), który zostanie anulowany [1]. Przy odbiorze należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].'}


Czas

In [13]:
print(f"Total time: {end-start:f}s")

Total time: 110.981640s


Zużycie VRAM

In [14]:
!nvidia-smi

Wed May 13 03:06:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             45W /   70W |   14497MiB /  15360MiB |     29%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Zużycie RAM

In [15]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       7.5Gi       2.0Gi        59Mi       3.2Gi       4.8Gi
Swap:             0B          0B          0B


**Odpowiedź:**  
Drobne różnice między wersjami modelu:
- `generation_config` wersji "chat" jest odrobinę krótszy (było to widoczne podczas pobierania przy 1. uruchomieniu),
- czas wykonania wersji "chat" również był odrobinę krótszy.  
  
Uzyskane odpowiedzi były bardzo podobne.

In [16]:
del pipeline, outputs
gc.collect()
torch.cuda.empty_cache()

In [17]:
try:
    pipeline = transformers.pipeline(
        "text-generation",
        model=model_name,
        model_kwargs={"dtype": torch.float16},
        device_map="cuda",
    )
    pipeline.model.eval()
    with torch.inference_mode():
        start = time.perf_counter()
        outputs = pipeline(
            messages,
            max_new_tokens=256,
        )
        end = time.perf_counter()
    print(outputs[0]["generated_text"][-1])
    print(f"Total time: {end-start:f}s")
except ValueError as e:
    print(e)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Could not load model CYFRAGOVPL/Llama-PLLuM-8B-chat with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/pipelines/base.py", line 232, in load_model
    model = model_class.from_pretrained(model, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 405, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 4245, in from_pretrained
    loading_info, disk_offload_index = cls._load_pretrained_model(model, state_dict, checkpoint_files, load_config)
   

Zużycie VRAM

In [18]:
!nvidia-smi

Wed May 13 03:07:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   74C    P0             31W /   70W |   14885MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Zużycie RAM

In [19]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.6Gi       364Mi        59Mi       7.8Gi       7.7Gi
Swap:             0B          0B          0B


**Odpowiedź:**  
Na darmowym planie Colab (T4 GPU z ok. 16GB VRAM) dochodzi do przekroczenia limitu pamięci.  
Opcja `"auto"` odpowiada za *offloading* nadmiaru danych do CPU.

ZADANIE #2 - Przepisz wyżej pokazany kod tak aby nie używać pipelinu, tylko natywnego podejścia transformers z użyciem takich komend jak AutoModelForCausalLM.from_pretrained(), AutoTokenizer.from_pretrained() i tokenizer.apply_chat_template(), oraz model.generate()

In [20]:
try:
    del pipeline, outputs
except:
    pass
gc.collect()
torch.cuda.empty_cache()

In [21]:
model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"

In [22]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [24]:
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

In [25]:
input_len = len(tokenized_chat["input_ids"][0])

In [26]:
model.eval()
with torch.inference_mode():
    start = time.perf_counter()
    outputs = model.generate(
        **tokenized_chat,
        max_new_tokens=256
    )
    end = time.perf_counter()

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [27]:
response = tokenizer.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)
print(response)

Aby odebrać paszport, należy osobiście zgłosić się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy posiadać ważny dokument tożsamości, a także aktualny dokument paszportowy, jeśli jest się w jego posiadaniu [1]. Ponadto, należy mieć przy sobie poprzedni paszport, który zostanie anulowany [1]. Przy odbiorze paszportu należy sprawdzić, czy dane osobowe i biometryczne są poprawne [1].


Czas

In [28]:
print(f"Total time: {end-start:f}s")

Total time: 127.017507s


Zużycie VRAM

In [29]:
!nvidia-smi

Wed May 13 03:11:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             49W /   70W |   14497MiB /  15360MiB |     41%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Zużycie RAM

In [30]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       7.6Gi       2.2Gi        59Mi       2.9Gi       4.7Gi
Swap:             0B          0B          0B


ZADANIE #3 - Do kodu napisanego w zadaniu 2 wprowadź optymalizacje obliczeń atencji np. flash-attention-2, aby zmniejszyć zużycie pamięci i szybkość generacji, opisz co zaobserwowałeś, czy coś się zmieniło i dlaczego nie, albo tak?

In [31]:
del tokenizer, model, tokenized_chat, outputs
gc.collect()
torch.cuda.empty_cache()

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [33]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [34]:
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

In [35]:
input_len = len(tokenized_chat["input_ids"][0])

In [36]:
model.eval()
with torch.inference_mode():
    start = time.perf_counter()
    outputs = model.generate(
        **tokenized_chat,
        max_new_tokens=256
    )
    end = time.perf_counter()

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [37]:
response = tokenizer.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)
print(response)

Aby odebrać paszport, należy osobiście udać się do punktu paszportowego, w którym złożono wniosek [1]. Przy odbiorze paszportu należy posiadać ważny dokument tożsamości, a także aktualny dokument paszportowy, jeśli się go posiada [1]. Ponadto, przy odbiorze paszportu należy również przedstawić poprzedni paszport, który zostanie anulowany [1]. Przy odbiorze paszportu należy również sprawdzić, czy dane osobowe i biometryczne są poprawne [1].


Czas

In [38]:
print(f"Total time: {end-start:f}s")

Total time: 137.431937s


Zużycie VRAM

In [39]:
!nvidia-smi

Wed May 13 03:14:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             50W /   70W |   14497MiB /  15360MiB |     41%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Zużycie RAM

In [40]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       4.4Gi       2.1Gi        59Mi       6.2Gi       7.9Gi
Swap:             0B          0B          0B


**Odpowiedź:**  
Próba zainstalowania `flash-attn` zakończyła się przekroczeniem limitu RAM darmowego planu Colab.  
Zastosowane zamiast tego SDPA nie poprawiło czasu wykonania, ale zauważalnie zmniejszyło zużycie RAM.  
Obliczenia nie zostały przyspieszone najpewniej z dwóch przyczyn:  
- SDPA największy wpływ ma przy dłuższych kontekstach (a tu mamy tylko 2 krótkie dokumenty),
- model ma *bottleneck* spowodowany byciem *offloaded* na CPU (za mało VRAM, nawet po optymalizacji).